- uv pip install sionna-rt

### Imports

In [1]:
import mitsuba as mi
mi.set_variant("cuda_ad_mono_polarized")   # força GPU
import sionna.rt
print("Variante atual:", mi.variant())
import drjit as dr
import matplotlib.pyplot as plt
import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, RadioMapSolver, PathSolver
from sionna.rt.utils import r_hat, subcarrier_frequencies

jitc_llvm_init(): LLVM API initialization failed ..


Variante atual: cuda_ad_mono_polarized


### Simple scene

#### Monostatic radar

In [7]:
no_preview = False

In [27]:
# ================================================================
# CONSTANTES
# ================================================================

C = 299_792_458.0

# ================================================================
# PARÂMETROS DO RADAR
# ================================================================

FC = 60e9                     # frequência [Hz]
LAMBDA = C / FC               # comprimento de onda [m]

PULSE_WIDTH = 20e-9           # duração do pulso [s]
PRI = 2e-6                    # Pulse Repetition Interval [s]
PRF = 1 / PRI                 # Pulse Repetition Frequency [Hz]

NUM_PULSES = 512              # número de pulsos

# Frequência de amostragem rápida
# Deve ser alta o suficiente para representar o pulso
FS = 10e9                     # 10 GHz

# Número de amostras dentro de um PRI
NUM_FAST_TIME = int(np.ceil(PRI * FS))

# ================================================================
# PARÂMETROS DERIVADOS
# ================================================================

RANGE_RESOLUTION = C * PULSE_WIDTH / 2

MAX_UNAMBIGUOUS_RANGE = C * PRI / 2

MAX_UNAMBIGUOUS_VELOCITY = LAMBDA * PRF / 4

DOPPLER_RESOLUTION = PRF / NUM_PULSES

VELOCITY_RESOLUTION = LAMBDA * DOPPLER_RESOLUTION / 2

COHERENT_PROCESSING_INTERVAL = NUM_PULSES * PRI


# ================================================================
# MOSTRAR PARÂMETROS
# ================================================================

print("=" * 65)
print("PARÂMETROS DO RADAR")
print("=" * 65)

print(f"Frequência                  : {FC/1e9:.2f} GHz")
print(f"Comprimento de onda         : {LAMBDA*1e3:.4f} mm")

print(f"\nDuração do pulso            : {PULSE_WIDTH*1e9:.2f} ns")
print(f"PRI                         : {PRI*1e6:.2f} us")
print(f"PRF                         : {PRF/1e3:.2f} kHz")

print(f"\nResolução de range          : {RANGE_RESOLUTION:.3f} m")
print(f"Range não ambíguo           : {MAX_UNAMBIGUOUS_RANGE:.3f} m")

print(f"\nNúmero de pulsos            : {NUM_PULSES}")
print(f"CPI                         : {COHERENT_PROCESSING_INTERVAL*1e3:.3f} ms")

print(f"Resolução Doppler           : {DOPPLER_RESOLUTION:.3f} Hz")
print(f"Resolução de velocidade     : {VELOCITY_RESOLUTION:.3f} m/s")

print(f"Velocidade não ambígua      : "
      f"±{MAX_UNAMBIGUOUS_VELOCITY:.3f} m/s")

print("=" * 65)


# ================================================================
# CARREGAR CENA
# ================================================================

scene = load_scene(
    sionna.rt.scene.simple_reflector,
    merge_shapes=False
)

# Frequência de operação
scene.frequency = FC

print("\nFrequência configurada no Sionna:",
      scene.frequency.numpy() / 1e9,
      "GHz")


# ================================================================
# MATERIAL DO REFLETOR
# ================================================================

reflector = scene.get("reflector")

print("\nMaterial original do refletor:")
print(reflector.radio_material)


# ------------------------------------------------
# OPCIONAL:
# substituir por material definido pelo usuário
# ------------------------------------------------

radar_material = RadioMaterial(
    name="target_material",
    thickness=0.01,
    relative_permittivity=5.0,
    conductivity=1e7
)

reflector.radio_material = radar_material

print("\nMaterial utilizado:")
print(reflector.radio_material)


# ================================================================
# VELOCIDADE DO ALVO
# ================================================================

reflector.velocity = [0, 0, -20]

print("\nVelocidade do alvo:")
print(reflector.velocity.numpy()[:, 0])


# ================================================================
# ARRAYS
# ================================================================

scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V"
)

scene.rx_array = scene.tx_array


# ================================================================
# RADAR MONOSTÁTICO
# ================================================================

RADAR_POSITION = [0, 0, 50]

scene.add(
    Transmitter(
        "tx",
        position=RADAR_POSITION,
        orientation=[0, 0, 0]
    )
)

scene.add(
    Receiver(
        "rx",
        position=RADAR_POSITION,
        orientation=[0, 0, 0]
    )
)


# ================================================================
# PATH SOLVER
# ================================================================

p_solver = PathSolver(deterministic=True)

paths = p_solver(
    scene=scene,

    # TX -> alvo -> RX
    max_depth=1,

    # Não queremos TX -> RX diretamente
    los=False,

    # Queremos reflexão especular
    specular_reflection=True,

    diffuse_reflection=False,
    refraction=False,
    diffraction=False
)


# ================================================================
# EXTRAIR PATHS
# ================================================================

tau = np.squeeze(paths.tau.numpy())
doppler = np.squeeze(paths.doppler.numpy())
valid = np.squeeze(paths.valid.numpy())

a_real = np.squeeze(paths.a[0].numpy())
a_imag = np.squeeze(paths.a[1].numpy())

a = a_real + 1j * a_imag


print("\nNúmero de caminhos:")
print(np.shape(tau))


# ================================================================
# SELECIONAR PATHS VÁLIDOS
# ================================================================

valid_indices = np.where(valid)[0]

print("\nPaths válidos:")
print(valid_indices)


for i in valid_indices:

    print("\n--------------------------------")
    print(f"Path {i}")

    print(
        f"Delay   : {tau[i]*1e9:.4f} ns"
    )

    print(
        f"Range   : {tau[i]*C/2:.4f} m"
    )

    print(
        f"Doppler : {doppler[i]:.4f} Hz"
    )

    print(
        f"|a|     : {np.abs(a[i]):.6e}"
    )


# ================================================================
# SELECIONAR O PRIMEIRO PATH VÁLIDO
# ================================================================

if len(valid_indices) == 0:
    raise RuntimeError("Nenhum caminho válido foi encontrado.")

path_index = valid_indices[0]

target_tau = tau[path_index]
target_doppler = doppler[path_index]
target_amplitude = a[path_index]

target_range = C * target_tau / 2


print("\n" + "=" * 65)
print("ALVO SELECIONADO")
print("=" * 65)

print(f"Range verdadeiro/RT : {target_range:.4f} m")
print(f"Delay               : {target_tau*1e9:.4f} ns")
print(f"Doppler RT          : {target_doppler:.4f} Hz")

print("=" * 65)


# ================================================================
# FAST TIME
# ================================================================

fast_time = np.arange(NUM_FAST_TIME) / FS


# ================================================================
# PULSO TRANSMITIDO
# ================================================================

pulse = np.where(
    fast_time < PULSE_WIDTH,
    1.0,
    0.0
)


# ================================================================
# ECO DE UM PULSO
# ================================================================

delay_samples = int(np.round(target_tau * FS))

echo = np.zeros(NUM_FAST_TIME, dtype=complex)


if delay_samples < NUM_FAST_TIME:

    end_sample = min(
        delay_samples + len(pulse),
        NUM_FAST_TIME
    )

    pulse_length = end_sample - delay_samples

    echo[
        delay_samples:end_sample
    ] = (
        target_amplitude
        * pulse[:pulse_length]
    )


# ================================================================
# MATCHED FILTER
# ================================================================

matched_filter = pulse[::-1]

range_profile = np.convolve(
    echo,
    matched_filter,
    mode="same"
)

range_axis = (
    np.arange(NUM_FAST_TIME) / FS
) * C / 2


# ================================================================
# PLOT RANGE PROFILE
# ================================================================

plt.figure(figsize=(10, 5))

plt.plot(
    range_axis,
    20 * np.log10(
        np.abs(range_profile) + 1e-12
    )
)

plt.axvline(
    target_range,
    linestyle="--",
    label="Range RT"
)

plt.xlim(
    0,
    MAX_UNAMBIGUOUS_RANGE
)

plt.xlabel("Range [m]")
plt.ylabel("Magnitude [dB]")
plt.title("Radar Range Profile")

plt.grid()
plt.legend()
plt.tight_layout()


# ================================================================
# PULSE-DOPPLER PROCESSING
# ================================================================

# Matriz:
#
#       fast time
#           ↓
# pulse 0   x x x x x
# pulse 1   x x x x x
# pulse 2   x x x x x
# ...
#
# Cada linha corresponde a um pulso.


radar_data = np.zeros(
    (NUM_PULSES, NUM_FAST_TIME),
    dtype=complex
)


# ================================================================
# GERAR TRENS DE PULSOS
# ================================================================

for p in range(NUM_PULSES):

    slow_time = p * PRI

    # fase Doppler
    doppler_phase = np.exp(
        1j * 2 * np.pi
        * target_doppler
        * slow_time
    )

    if delay_samples < NUM_FAST_TIME:

        end_sample = min(
            delay_samples + len(pulse),
            NUM_FAST_TIME
        )

        pulse_length = end_sample - delay_samples

        radar_data[
            p,
            delay_samples:end_sample
        ] = (
            target_amplitude
            * doppler_phase
            * pulse[:pulse_length]
        )


# ================================================================
# MATCHED FILTER EM CADA PULSO
# ================================================================

range_data = np.zeros_like(radar_data)


for p in range(NUM_PULSES):

    range_data[p, :] = np.convolve(
        radar_data[p, :],
        matched_filter,
        mode="same"
    )


# ================================================================
# DOPPLER FFT
# ================================================================

doppler_data = np.fft.fftshift(
    np.fft.fft(
        range_data,
        axis=0
    ),
    axes=0
)


# ================================================================
# EIXO DE DOPPLER
# ================================================================

doppler_axis = np.fft.fftshift(
    np.fft.fftfreq(
        NUM_PULSES,
        d=PRI
    )
)


# ================================================================
# EIXO DE VELOCIDADE
# ================================================================

velocity_axis = (
    doppler_axis * LAMBDA / 2
)


# ================================================================
# NORMALIZAR RANGE-DOPPLER
# ================================================================

range_doppler_db = (
    20 * np.log10(
        np.abs(doppler_data)
        + 1e-12
    )
)

range_doppler_db -= np.max(
    range_doppler_db
)


# ================================================================
# RANGE-DOPPLER MAP
# ================================================================

plt.figure(figsize=(11, 6))

plt.imshow(
    range_doppler_db,
    aspect="auto",
    origin="lower",
    extent=[
        range_axis[0],
        range_axis[-1],
        velocity_axis[0],
        velocity_axis[-1]
    ]
)

plt.colorbar(
    label="Magnitude [dB]"
)

plt.xlabel("Range [m]")
plt.ylabel("Radial velocity [m/s]")

plt.title(
    "Range-Doppler Map"
)

plt.xlim(
    0,
    MAX_UNAMBIGUOUS_RANGE
)

plt.tight_layout()


# ================================================================
# ENCONTRAR PICO
# ================================================================

peak_index = np.unravel_index(
    np.argmax(np.abs(doppler_data)),
    doppler_data.shape
)

peak_pulse_index = peak_index[0]
peak_range_index = peak_index[1]

estimated_doppler = (
    doppler_axis[peak_pulse_index]
)

estimated_velocity = (
    velocity_axis[peak_pulse_index]
)

estimated_range = (
    range_axis[peak_range_index]
)


print("\n" + "=" * 65)
print("ESTIMATIVA DO RADAR")
print("=" * 65)

print(
    f"Range estimado       : "
    f"{estimated_range:.3f} m"
)

print(
    f"Range RT             : "
    f"{target_range:.3f} m"
)

print(
    f"Doppler estimado     : "
    f"{estimated_doppler:.3f} Hz"
)

print(
    f"Doppler Sionna       : "
    f"{target_doppler:.3f} Hz"
)

print(
    f"Velocidade estimada  : "
    f"{estimated_velocity:.3f} m/s"
)

print("=" * 65)


# ================================================================
# VISUALIZAÇÃO SIONNA
# ================================================================

cam = Camera(
    position=[0, 100, 50],
    look_at=[0, 0, 30]
)

scene.preview(paths=paths)

PARÂMETROS DO RADAR
Frequência                  : 60.00 GHz
Comprimento de onda         : 4.9965 mm

Duração do pulso            : 20.00 ns
PRI                         : 2.00 us
PRF                         : 500.00 kHz

Resolução de range          : 2.998 m
Range não ambíguo           : 299.792 m

Número de pulsos            : 512
CPI                         : 1.024 ms
Resolução Doppler           : 976.562 Hz
Resolução de velocidade     : 2.440 m/s
Velocidade não ambígua      : ±624.568 m/s

Frequência configurada no Sionna: [60.000004] GHz

Material original do refletor:
ITURadioMaterial type=metal
                 eta_r=1.000
                 sigma=10000000.000
                 thickness=0.010
                 scattering_coefficient=0.000
                 xpd_coefficient=0.000

Material utilizado:
RadioMaterial eta_r=5.000
              sigma=10000000.000
              thickness=0.010
              scattering_coefficient=0.000
              xpd_coefficient=0.000

Velocidade do alvo:


ValueError: Calling nonzero on 0d arrays is not allowed. Use np.atleast_1d(scalar).nonzero() instead. If the context of this error is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.

In [24]:
# ================================================================
# CONFIGS radar
# ================================================================
FC = 60e9               # 60 GHz
C = 299792458           # velocidade da luz [m/s]

PULSE_WIDTH = 20e-9    # 20 ns
PRI = 2e-6             # 2 us
PRF = 1/PRI            # Hz
NUM_PULSES = 128
R_UNANB = C*PRI/2


print("Radar parameters: \n",
    "Type: monostatic pulsed radar\n"
    "Operation Frequency: ", FC, " Hz\n"
    "Pulse_width: ", PULSE_WIDTH, " s\n",
    "Bandwith: ", PRI, " m\n" #############################
    "PRI: ", PRI, " s\n",
    "PRF: ", PRF, " Hz\n",
    "Num_pulses: ", NUM_PULSES, "\n",
    "PRF: ", PRF, " Hz\n",
    "Unambiguos range: ", R_UNANB, " m\n"
    "Range_resolution: ", R_UNANB, " m\n" ################################
    "Unambiguos range: ", R_UNANB, " m\n"
    "Unambiguos range: ", R_UNANB, " m\n"
    "Unambiguos range: ", R_UNANB, " m\n")
     

# ================================================================
# CARREGAR CENA
# ================================================================
scene = load_scene(
    sionna.rt.scene.simple_reflector,
    merge_shapes=False)

scene.frequency = FC

print("Frequência:", scene.frequency.numpy() / 1e9, "GHz")
print("Comprimento de onda:", scene.wavelength.numpy()[0], "m")

# ================================================================
# REFLETOR / ALVO
# ================================================================

reflector = scene.get("reflector")
print("Velocidade inicial do alvo: ", reflector.velocity.numpy()[:, 0])

# Velocidade do alvo
reflector.velocity = [0, 0, -20]

print("Velocidade após atualização: ", reflector.velocity.numpy()[:, 0])

# ================================================================
# ANTENNAS
# ================================================================

scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V")
scene.rx_array = scene.tx_array

# ================================================================
# RADAR MONOSTÁTICO
# ================================================================
RADAR_POSITION = [0, 0, 50]
# Posição do radar (TX = RX)
scene.add(Transmitter(
    "tx",
    position=RADAR_POSITION,
    orientation=[0, 0, 0]))
scene.add(Receiver(
    "rx",
    position=RADAR_POSITION,
    orientation=[0, 0, 0]))

# ================================================================
# RT
# ================================================================

p_solver = PathSolver()

paths = p_solver(
    scene=scene,
    max_depth=1, # TX -> target -> RX
    los=False, # Monostatico
    specular_reflection=True,
    diffuse_reflection=True,
    refraction=True,
    diffraction=True
)

# ================================================================
# Paths
# ================================================================

print("Número de caminhos:", paths.tau.numpy().shape[-1])

print("Interações: ",paths.interactions.numpy())

# ================================================================
# EXTRAIR CAMINHOS VÁLIDOS
# ================================================================

tau = paths.tau.numpy().reshape(-1)
doppler = paths.doppler.numpy().reshape(-1)
valid = paths.valid.numpy().reshape(-1)

print("\n--- CAMINHOS VÁLIDOS ---")

for i in range(len(tau)):

    if not valid[i]:
        continue
    print(f"\nPath {i}")
    print(f"  Delay   : {tau[i]*1e9:.4f} ns")
    print(f"  Range   : {tau[i] * C / 2:.4f} m")
    print(f"  Doppler : {doppler[i]:.4f} Hz")

# ================================================================
# VISUALIZAÇÃO
# ================================================================

cam = Camera(
    position=[0, 100, 50],
    look_at=[0, 0, 30]
)

scene.preview(paths=paths)

Radar parameters: 
 Type: monostatic pulsed radar
Operation Frequency:  60000000000.0  Hz
Pulse_width:  2e-08  s
 Bandwith:  2e-06  m
PRI:  2e-06  s
 PRF:  500000.0  Hz
 Num_pulses:  128 
 PRF:  500000.0  Hz
 Unambiguos range:  299.792458  m
Range_resolution:  299.792458  m
Unambiguos range:  299.792458  m
Unambiguos range:  299.792458  m
Unambiguos range:  299.792458  m

Frequência: [60.000004] GHz
Comprimento de onda: 0.0049965405 m
Velocidade inicial do alvo:  [0. 0. 0.]
Velocidade após atualização:  [  0.   0. -20.]
Número de caminhos: 1
Interações:  [[[[1]]]]

--- CAMINHOS VÁLIDOS ---

Path 0
  Delay   : 333.5641 ns
  Range   : 50.0000 m
  Doppler : -8005.5391 Hz


In [18]:
# Load scene with a single reflector
scene = load_scene(
    sionna.rt.scene.simple_reflector,
    merge_shapes=False)

print(scene.objects)
# Inspect the velocity of this object
print("Velocity vector: ", scene.get("reflector").velocity.numpy()[:,0])
# Update velocity vector
scene.get("reflector").velocity = [0, 0, -20]
print("Velocity vector after update: ", scene.get("reflector").velocity.numpy()[:,0])

# Configure arrays for all transmitters and receivers in the scene
scene.tx_array = PlanarArray(num_rows=1,num_cols=1,pattern="iso", polarization="V")
scene.rx_array = scene.tx_array

# Add a transmitter and a receiver
scene.add(Transmitter(
    "tx", 
    position= [0, 0, 50], 
    orientation=[0,0,0]))

scene.add(Receiver(
    "rx", 
    position= [0, 0, 50], 
    orientation=[0,0,0]))

# Compute paths
p_solver = PathSolver()
paths = p_solver(
    scene=scene, 
    max_depth=1)

# Visualize the scene and propagation paths
if no_preview:
    cam = Camera(position=[0, 100, 50], look_at=[0,0,30])
    scene.render(camera=cam, paths=paths);
else:
    scene.preview(paths=paths)

{'reflector': <sionna.rt.scene_object.SceneObject object at 0x000001E134402D50>}
Velocity vector:  [0. 0. 0.]
Velocity vector after update:  [  0.   0. -20.]


#### Bistatic radar

### Codebook creation

### MIMO Simple Scene

#### Monostatic radar using multiple beams

#### Bistatic radar using multiple beams

### MIMO Simple urban scenario

#### Monostatic radar

#### Bistatic radar

In [4]:
no_preview = False
scene = load_scene(
    sionna.rt.scene.simple_street_canyon_with_cars,
    merge_shapes=False)
cam = Camera(
    position=[50,0,130],
    look_at=[10,0,0])

# Parameters for ray tracing
max_depth = 3
refraction = False # Toggle to true to see the impact of refraction
diffraction = False # Toggle to true to see the impact of diffraction

# coonfigure antennas
scene.tx_array = PlanarArray(
    num_rows=1, 
    num_cols=1, 
    pattern="tr38901", 
    polarization="V")
scene.rx_array = scene.tx_array

# place tx and rx
hight = np.array([0, 0, 1.15])

car_2 = scene.get("car_2")
scene.add(Transmitter(
    "tx", 
    position= car_2.position.numpy()[:,0] + hight, #[25, 5.6, 1.9], 
    orientation=[np.pi,0,0]))
car_3 = scene.get("car_3")
scene.add(Receiver(
    "rx", 
    position=car_3.position.numpy()[:,0] + hight, 
    orientation=[0,0,0]))

# scene.objects

if no_preview:
    scene.render(camera=cam);
else:
    scene.preview();

In [ ]:
# Configure an OFDM resource grid
num_ofdm_symbols = 25
num_subcarriers = 1024
subcarrier_spacing = 30e3
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)
ofdm_symbol_duration = 1/subcarrier_spacing

# Define a velocity vector and the corresponding displacement over the duration
# of one OFDM symbol
velocity_vec = np.array([10,0,0])
displacement_vec = velocity_vec*ofdm_symbol_duration

# Assign velocity vector to cars driving in -x direction
for j in range(1,6):
    scene.get(f"car_{j}").velocity = -velocity_vec

# Assign velocity vector to cars driving in x direction
for j in range(6,9):
    scene.get(f"car_{j}").velocity = velocity_vec

# Compute paths
scene.get("tx").velocity = -velocity_vec
scene.get("rx").velocity = velocity_vec

p_solver = PathSolver()
paths = p_solver(
    scene=scene, 
    max_depth=max_depth, 
    refraction=refraction, 
    diffraction=diffraction)

# Compute the corresponding channel frequency responses with time evolution
h_dop = paths.cfr(
    frequencies=frequencies,
    sampling_frequency=1/ofdm_symbol_duration,
    num_time_steps=num_ofdm_symbols,
    normalize_delays=False, out_type="numpy")
h_dop = np.squeeze(h_dop)




# paths = p_solver(scene=scene, 
#                  max_depth=1)
# print("Path interaction (0=LoS, 1=Specular reflection): ", paths.interactions.numpy())
# print("Doppler shifts (Hz): ", paths.doppler.numpy())

if no_preview:
    scene.render(camera=cam, 
                 paths=paths);
else:
    scene.preview(paths=paths)

Path interaction (0=LoS, 1=Specular reflection):  [[[[0 1 1 1]]]]
Doppler shifts (Hz):  [[[0. 0. 0. 0.]]]


### move

In [ ]:
car_2 = scene.get("car_2")
print("Position: ", car_2.position.numpy()[:,0])
print("Orientation: ", car_2.orientation.numpy()[:,0])

# Move the car 10m along the y-axis
#car_2.position += [0, 10, 0]
# And rotate it by 90 degree around the z-axis
#car_2.orientation = [np.pi/2, 0, 0]
if no_preview:
    scene.render(camera=cam);
else:
    scene.preview();

Position:  [25.         5.5999994  0.7500001]
Orientation:  [0. 0. 0.]


### next steps

In [ ]:
# Montar slide
# falar dos avanços, que consistirar em referencial teórico de radar, simulações com radar monostático e bistático utilizando sionna


# pretensões

# Não será necessário utilizar simulador de FEM, a idea permanece em utilizar o Sionna e o Wireless insite
# Estou fechando melhor o contexto da atividade desenvolvida, antes estava entre imagem e localização, agora vai fechar em localização com radar bistático localmente separado, 
# Localização de emissores em cenários V2V utilizando sensing baseado em formas de onda 5G NR e aprendizado multimodal a partir de canais gerados por ray tracing. 
# Enlace V2V cuja forma de onda simultaneamente realiza comunicação e sensing (radar bistático espacialmente separado). 
# O radar seria de tipo ativo, onde haveria a comunicação v2v e, com essa comunicação seria possível obter a posição e velocidade deste carro.
# A atividade se classifica como localização do transmissor. (Emitter localization/passive emiter localization/ bistatic/multistatic localization)
# Comunicação + Sensing + aprendizado de máquina + ray tracing + visão computacional + V2V. 

# Normalmente os métodos que utilizam na literatura consistem em separar os caminhos de los e nlos1 e a apartir disso inferir a posição do Tx, 
# Obs:
# No radar bistático espacialmente separado conseguimos obter a distância do path completo, ângulo de chegada, doppler e magnitude, (às vezes se sabe o ângulo de partida também). 
# Normalmente se sabe a posição do Tx, do Rx, de seus respectivos ângulos de transmissão e recepção, se foca no caminho de LoS e nos caminhos NLoS-1 (uma interação apenas) e, com isso 
# dá pra obter a posição do target de 1 interação.

# A questão de tentar obter a posição do Tx é que precisamos conhecer o cenário, do contrário só parece que o transmissor está em várias posições e distâncias (por conta dos multipaths). 
# Dessa maneira, o diferencial é que poderíamos colocar os dados do radar em um modelo e utilizar isso como input para obter essas coordenadas e velocidade.

# Poderíamos ainda comparar com a localização produzida pelo próprio radar sem ML;
# Visão computacional (apesar de que processamento de sensores, como radar já qualifica como visão computacional): 
# poderíamos ainda colocar imagens do ponto de vista do receptor como entrada do modelo para servir de informação adicional da geometria do cenário, e assim reduzir a ambiguidade de localização produzida por radar bistático.
# Podemos comparar os canais gerados pelo WI e Sionna, em como a escolha do simulador de ray tracing influencia o desempenho de um sistema de sensing baseado em aprendizado de máquina.












# Pretendo utilizar codebook 3gpp (Verificar o codebook 3GPP)
# Integração do Sionna com o Raymobtime
# utilizar o Raymobtime para produzir as imagens e produzir um dataset substancial de canais mimo v2v,
# Pretendo processar no matlab utilizando a 5G toolbox